# Step 2 Demo — Hemolytic / Toxicity Filter

Score candidate peptides for host toxicity and split them into toxin / non-toxin sets.

This notebook:
1. loads a tiny Step-1 style CSV
2. runs the pretrained FusionPeptide toxicity model (`mode=101`)
3. visualizes toxicity probabilities


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

STEP_DIR = Path('.').resolve()
print('Working directory:', STEP_DIR)
assert (STEP_DIR / 'infer_toxin.py').exists(), 'Please open/run this notebook from 02_toxicity_filter/'


In [ ]:
INPUT_CSV = STEP_DIR / 'examples' / 'sample_input.csv'
OUT_DIR = STEP_DIR / 'outputs' / 'notebook_demo'
THRESHOLD = 0.5

print('Input:', INPUT_CSV)
print(pd.read_csv(INPUT_CSV, header=None, names=['seq','copy']).head())


In [ ]:
from infer_toxin import run_inference
from types import SimpleNamespace

args = SimpleNamespace(
    csv=str(INPUT_CSV),
    out_dir=str(OUT_DIR),
    model_path=str(STEP_DIR / 'checkpoints' / 'model_1.pth'),
    threshold=THRESHOLD,
    max_len=30,
    batch_size=32,
    q_encoder='gru',
    v_encoder='resnet34',
    channels=32,
    mode='101',
    cpu=True,
)

toxin_path, non_toxin_path, all_path = run_inference(args)


In [ ]:
all_df = pd.read_csv(all_path)
print(all_df[['sequence','tox_prob']].describe())
display(all_df.sort_values('tox_prob', ascending=False).head())

plt.figure(figsize=(6,3.5))
plt.hist(all_df['tox_prob'], bins=20, color='#2c7fb8', edgecolor='white')
plt.axvline(THRESHOLD, color='#d7301f', linestyle='--', label=f'threshold={THRESHOLD}')
plt.xlabel('Toxicity probability')
plt.ylabel('Count')
plt.title('Step 2 toxicity score distribution')
plt.legend()
plt.tight_layout()
plt.show()

print('Non-toxins:', non_toxin_path)
print(pd.read_csv(non_toxin_path).head())


## Next step

Use the non-toxin CSV in **Step 3**:

```bash
cd ../03_structure_prediction
# conda activate helix
python infer_batch.py --csv_file ../02_toxicity_filter/outputs/notebook_demo/sample_non_toxins.csv --output_dir outputs/pdb/demo
```
